# **Spam Detection using Naive Bayes**

In [1]:
import numpy as np

# **Collecting the Dataset**

In [2]:
spam = [
    "To use your credit, click the new WAP link in the next years txt message or click here", 
    "Thanks for your subscription to New Ringtone UK your new mobile will be charged £5/month Please confirm annoncement by replying", 
    "As a valued customer, I am pleased to advise you that following recent delivery waiting review of your Mob No. you are awarded with. Call us to review.", 
    "Please call our new customer service representative on", 
    "We are trying to contact you. Last weekends customer draw shows that you won a £1000 prize GUARANTEED. Calling years", 
]

In [3]:
# leave one sentence from testing our model later
Spam_test =  ["Customer service annoncement. You have a New Years delivery waiting for you. click"]

In [4]:
non = [
    "I don't think he goes to usf, he lives around here though", 
    "New car and house for my parents. i have only new job in hand", 
    "Great escape. I fancy the bridge but needs her lager. See you tomorrow", 
    "Tired. I haven't slept well the past few nights.",
    "Too late. I said i have the website. I didn't i have or dont have the slippers", 
    "I might come by tonight then if my class lets out early", 
    "Jos ask if u wana meet up?", 
    "That would be great. We'll be at the Guild. We can try meeting with the customer on Bristol road or somewhere"
    ]

In [5]:
# another sentence from non for testing our model
Spam_test_2 = ["That would be great. We'll be at the Guild. We can try meeting with the customer on Bristol road or somewhere "]

# **Basic Pre-Processing**

In [6]:
# !pip install gensim

In [7]:
from gensim.parsing.preprocessing import remove_stopwords
from gensim.parsing.porter import PorterStemmer
from gensim.utils import tokenize

In [8]:
test_sentence = non[4]
test_sentence = non[5]
test_sentence = spam[1]

print(test_sentence)

remove_stops = remove_stopwords(test_sentence)
print(remove_stops)

p = PorterStemmer()
stemmed = p.stem(remove_stops)
print(stemmed)

tokens = list(tokenize(stemmed))
print(len(tokens))

Thanks for your subscription to New Ringtone UK your new mobile will be charged £5/month Please confirm annoncement by replying
Thanks subscription New Ringtone UK new mobile charged £5/month Please confirm annoncement replying
thanks subscription new ringtone uk new mobile charged £5/month please confirm annoncement repli
13


# **Create a dictionary of words**

In [9]:
def tokenize_sentence(sentence):
    p = PorterStemmer()
    remove_stops = remove_stopwords(sentence)
    stemmed = p.stem(remove_stops)
    tokens = tokenize(stemmed)
    return list(tokens)

In [10]:
dictionary = set()      # will have unique values only
spam_tokenized = []
nons_tokenized = []

for sentence in spam:
    sentence_tokens = tokenize_sentence(sentence)
    spam_tokenized.append(sentence_tokens)
    dictionary = dictionary.union(sentence_tokens)       # add sentence words to the dictionary

for sentence in non:
    sentence_tokens = tokenize_sentence(sentence)
    nons_tokenized.append(sentence_tokens)
    dictionary = dictionary.union(sentence_tokens)
    
print("tokenize spam: ",spam_tokenized)
print("tokenize non: ",nons_tokenized)
print("Dictionary: ",dictionary)
    

tokenize spam:  [['to', 'use', 'credit', 'click', 'new', 'wap', 'link', 'years', 'txt', 'message', 'click'], ['thanks', 'subscription', 'new', 'ringtone', 'uk', 'new', 'mobile', 'charged', 'month', 'please', 'confirm', 'annoncement', 'repli'], ['as', 'valued', 'customer', 'i', 'pleased', 'advise', 'following', 'recent', 'delivery', 'waiting', 'review', 'mob', 'no', 'awarded', 'with', 'call', 'review'], ['please', 'new', 'customer', 'service', 'repres'], ['we', 'trying', 'contact', 'you', 'last', 'weekends', 'customer', 'draw', 'shows', 'won', 'prize', 'guaranteed', 'calling', 'year']]
tokenize non:  [['i', 'don', 't', 'think', 'goes', 'usf', 'l'], ['new', 'car', 'house', 'parents', 'new', 'job', 'hand'], ['great', 'escape', 'i', 'fancy', 'bridge', 'needs', 'lager', 'see', 'tomorrow'], ['tired', 'i', 'haven', 't', 'slept', 'past', 'nights'], ['too', 'late', 'i', 'said', 'website', 'i', 'didn', 't', 'dont', 'slipp'], ['i', 'come', 'tonight', 'class', 'lets', 'earli'], ['jos', 'ask', 'u',

# **Basic Stats**

In [11]:
# These things do not depend on an individual word so let's calculate them separately

total_word_count = len(dictionary)
total_spam_messages = len(spam_tokenized)
total_all_messages = len(spam_tokenized) + len(nons_tokenized)

print("Total Number of Words: ", total_word_count)

Total Number of Words:  101


In [12]:
# p(spam) ... does not depend on an individual word so let's calculate that separately

p_spam = total_spam_messages / total_all_messages

print("P(spam) = ", p_spam)

P(spam) =  0.38461538461538464


In [13]:
# Helper function to count occurances

def count_word_in_messages(word, messages):
    total_count = 0
    for msg in messages:
        if word in msg:         # notice this ensured uniqueness automatically
            total_count += 1 
        
        
    return total_count

# **The Actual Probability Computation**

In [14]:
final_prob = 1      # can't start from 0

for test_sentence in Spam_test:
    test_sentence = tokenize_sentence(test_sentence)
    print(test_sentence)
    
    # let's run this for each word separately
    for word in test_sentence:
        print("-----------------")
        print("Running for word: ", word)
        
        # Find P(W|spam)
        spam_count = count_word_in_messages(word, spam_tokenized)
        p_w_spam = spam_count / total_spam_messages
        print("P(W | spam) = ",p_w_spam)
        
        # Find P(w)
        w_count = count_word_in_messages(word, spam_tokenized)
        w_count += count_word_in_messages(word, nons_tokenized)
        p_w = w_count / total_all_messages
        print("P(w)     = ", p_w)
        
        # Find P(Spam| w)
        p_spam_w = (p_w_spam * p_spam) / p_w
        print("P(spam) = ", p_spam)
        print("P(spam | w) = ", p_spam_w)
        print("")
        final_prob *= p_spam_w
        
    print("P(spam | all_words ) = ",final_prob)

['customer', 'service', 'annoncement', 'you', 'new', 'years', 'delivery', 'waiting', 'you', 'click']
-----------------
Running for word:  customer
P(W | spam) =  0.6
P(w)     =  0.3076923076923077
P(spam) =  0.38461538461538464
P(spam | w) =  0.75

-----------------
Running for word:  service
P(W | spam) =  0.2
P(w)     =  0.07692307692307693
P(spam) =  0.38461538461538464
P(spam | w) =  1.0

-----------------
Running for word:  annoncement
P(W | spam) =  0.2
P(w)     =  0.07692307692307693
P(spam) =  0.38461538461538464
P(spam | w) =  1.0

-----------------
Running for word:  you
P(W | spam) =  0.2
P(w)     =  0.07692307692307693
P(spam) =  0.38461538461538464
P(spam | w) =  1.0

-----------------
Running for word:  new
P(W | spam) =  0.6
P(w)     =  0.3076923076923077
P(spam) =  0.38461538461538464
P(spam | w) =  0.75

-----------------
Running for word:  years
P(W | spam) =  0.2
P(w)     =  0.07692307692307693
P(spam) =  0.38461538461538464
P(spam | w) =  1.0

-----------------
Runn

## What is the reason of using logs? 
*The reason is because while probability can much smaller means near zero which can be consider zero by computers sometime which is known as underflow so when we take logs it can calcualate the value for every smaller values and not consider it zero and at end you can take antilog and get the same result you want it gives more reliable answer then simply method*

In [18]:
import math

for test_sentence in Spam_test:
    test_sentence = tokenize_sentence(test_sentence)
    print(test_sentence)

    # Start from 0 because log(1) = 0 (neutral for addition)
    final_log_prob = 0

    for word in test_sentence:
        print("-----------------")
        print("Running for word:", word)

        # P(W | Spam)
        spam_count = count_word_in_messages(word, spam_tokenized)
        p_w_spam = spam_count / total_spam_messages

        # P(W)
        w_count = count_word_in_messages(word, spam_tokenized) + count_word_in_messages(word, nons_tokenized)
        p_w = w_count / total_all_messages

        # P(Spam)

        # Log-space calculation:
        if p_w_spam > 0 and p_w > 0 and p_spam > 0:
            log_p_w_spam = math.log(p_w_spam)
            log_p_spam = math.log(p_spam)
            log_p_w = math.log(p_w)
            
            print("log(P(W | spam)) =", p_w_spam)
            print("log(P(W)) =", p_w)
            print("log(P(Spam)) =", p_spam)

            log_p_spam_given_w = log_p_w_spam + log_p_spam - log_p_w
            print("log(P(Spam | W)) =", log_p_spam_given_w)

            final_log_prob += log_p_spam_given_w
        else:
            print("Skipped word due to zero probability")

        # print("Current Total Log Probability =", final_log_prob)
        print("")

    # Convert log prob back to normal scale if you want
    print("Final log(P(spam | all_words)):", final_log_prob)
    print("Final P(spam | all_words):", math.exp(final_log_prob))


['customer', 'service', 'annoncement', 'you', 'new', 'years', 'delivery', 'waiting', 'you', 'click']
-----------------
Running for word: customer
log(P(W | spam)) = 0.6
log(P(W)) = 0.3076923076923077
log(P(Spam)) = 0.38461538461538464
log(P(Spam | W)) = -0.287682072451781

-----------------
Running for word: service
log(P(W | spam)) = 0.2
log(P(W)) = 0.07692307692307693
log(P(Spam)) = 0.38461538461538464
log(P(Spam | W)) = 0.0

-----------------
Running for word: annoncement
log(P(W | spam)) = 0.2
log(P(W)) = 0.07692307692307693
log(P(Spam)) = 0.38461538461538464
log(P(Spam | W)) = 0.0

-----------------
Running for word: you
log(P(W | spam)) = 0.2
log(P(W)) = 0.07692307692307693
log(P(Spam)) = 0.38461538461538464
log(P(Spam | W)) = 0.0

-----------------
Running for word: new
log(P(W | spam)) = 0.6
log(P(W)) = 0.3076923076923077
log(P(Spam)) = 0.38461538461538464
log(P(Spam | W)) = -0.287682072451781

-----------------
Running for word: years
log(P(W | spam)) = 0.2
log(P(W)) = 0.07692

Perfect, Abdullah! Here's a clear and well-structured **summary** you can add to your Markdown to explain the transition from using **individual conditional probabilities** to the more accurate **joint probability** approach.

---

## 📌 Summary: From Individual Conditional Probabilities to Joint Probability

### 🔹 **Initial Approach (Using Conditional Probability for Each Word Individually)**

In the beginning, we assumed that each word in the email is **independent** of the others. This led to using **conditional probability** for each word separately:

$$
P(\text{Spam} \mid \text{word1}) = \frac{P(\text{word1} \mid \text{Spam}) \cdot P(\text{Spam})}{P(\text{word1})}
$$

$$
P(\text{Spam} \mid \text{word2}) = \frac{P(\text{word2} \mid \text{Spam}) \cdot P(\text{Spam})}{P(\text{word2})}
$$

To combine multiple words, we multiplied these individual results (assuming independence):

$$
P(\text{Spam} \mid \text{word1, word2}) \approx \frac{P(\text{word1} \mid \text{Spam}) \cdot P(\text{word2} \mid \text{Spam}) \cdot P(\text{Spam})^2}{P(\text{word1}) \cdot P(\text{word2})}
$$

### 🔍 **Intuition Behind It:**

* Each word contributes to the overall likelihood of spam.
* The total probability is built by multiplying the effect of each word.
* This works **only if** the words are independent (which is rarely true in real emails).

---

### 🔹 **Improved Approach (Using Joint Probability)**

Later, we switched to using the **joint probability** of observing both words together, as a single event:

$$
P(\text{Spam} \mid \text{word1, word2}) = 
\frac{P(\text{word1, word2} \mid \text{Spam}) \cdot P(\text{Spam})}
{P(\text{word1, word2})}
$$

### 🔍 **Intuition Behind It:**

* Instead of treating words separately, we look at the **combined occurrence** of word1 and word2 as a single observation.
* This gives a more **realistic estimate**, since words often appear together (e.g., "free" and "win").
* The denominator $P(\text{word1, word2})$ reflects the **actual probability** of seeing both words in the same email, which gives us more accurate spam detection.

---

Let me know when you're ready to build the final algorithm based on this foundation.


# 🧠 Intuition Summary
**In your method, denominator is a product of individual probabilities, like:**

"What's chance word1 appears in any email?" × "What's chance word2 appears in any email?"

**In joint method, denominator is the chance that:**

"Both word1 and word2 appear together in an email" — which exactly matches what you observed!



# **the algorithum is given the file name Multinomial Bayesian Algorithum**

In [15]:
# ------------------- by chatgpt 

import math

for test_sentence in Spam_test:
    test_sentence = tokenize_sentence(test_sentence)
    print(test_sentence)

    # Initialize log-probability with prior probability P(Spam)
    final_prob = math.log(p_spam)

    # Let's run this for each word separately
    for word in test_sentence:
        print("-----------------")
        print("Running for word:", word)

        # Laplace smoothing parameters
        alpha = 1  # Smoothing factor
        vocab_size = len(set(word for msg in spam_tokenized + nons_tokenized for word in msg))  # Unique words

        # Find P(W|spam) with Laplace smoothing
        spam_count = count_word_in_messages(word, spam_tokenized)
        p_w_spam = (spam_count + alpha) / (total_spam_messages + alpha * vocab_size)
        print("P(W | spam) =", p_w_spam)

        # Find P(W) with Laplace smoothing
        w_count = count_word_in_messages(word, spam_tokenized) + count_word_in_messages(word, nons_tokenized)
        p_w = (w_count + alpha) / (total_all_messages + alpha * vocab_size)
        print("P(W) =", p_w)

        # Find P(Spam | W) using log probabilities
        final_prob += math.log(p_w_spam) - math.log(p_w)

    # Convert log probability back to normal scale
    print("P(spam | all_words) =", math.exp(final_prob))


['customer', 'service', 'annoncement', 'you', 'new', 'years', 'delivery', 'waiting', 'you', 'click']
-----------------
Running for word: customer
P(W | spam) = 0.03773584905660377
P(W) = 0.043859649122807015
-----------------
Running for word: service
P(W | spam) = 0.018867924528301886
P(W) = 0.017543859649122806
-----------------
Running for word: annoncement
P(W | spam) = 0.018867924528301886
P(W) = 0.017543859649122806
-----------------
Running for word: you
P(W | spam) = 0.018867924528301886
P(W) = 0.017543859649122806
-----------------
Running for word: new
P(W | spam) = 0.03773584905660377
P(W) = 0.043859649122807015
-----------------
Running for word: years
P(W | spam) = 0.018867924528301886
P(W) = 0.017543859649122806
-----------------
Running for word: delivery
P(W | spam) = 0.018867924528301886
P(W) = 0.017543859649122806
-----------------
Running for word: waiting
P(W | spam) = 0.018867924528301886
P(W) = 0.017543859649122806
-----------------
Running for word: you
P(W | spa